# V60 - Bradley-Terry module strengths

Pre-registered correction with full disclosure: v59 and its follow-up probe were vacuous. Both built pair differences from stacked copies of a single user row, so every difference vector was identically zero (verified: variance 0.0). The MLP 0.5000 and the irreproducible 0.615 findings are withdrawn, not cited. With one row per user, the only coherent pairwise model uses module-identity differences, which is Bradley-Terry: 17 strengths fit from within-user pairs. If BT cannot beat popularity under grouped folds, the pairwise family is closed with evidence instead of armchair claims.

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer

SEED = 202
N_SPLITS = 5
GATE_FLOOR = 0.002
rng = np.random.RandomState(SEED)

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v60")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve())

DATA: /home/athilla/Documents/IF_ITB/Lomba/IT-TODAY/intelligo-course-recommender/data


## Zero-diff proof, then Bradley-Terry

First assert the v59 bug class is gone: user-row differences are zero by construction, so only module-identity differences enter. BT fits 17 strengths per fold from informative pairs.

In [2]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train = len(train)
J = len(MODS)

by_user = chats.groupby("user_id")["user_chat_text"].apply(list)

def template_group(texts):
    if not texts:
        return "NO_CHAT"
    return re.sub(r"\d+", "#", " ".join(str(texts[0]).lower().split()[:6]))
fold_groups = np.array([template_group(by_user.get(u, [])) for u in train["user_id"]])
strata = Y.argmax(1)

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

EI = np.eye(J)
oof_bt = np.zeros_like(Y)
fids = np.zeros(n_train, int)
skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
for fi, (tr, va) in enumerate(skf.split(Y, strata, fold_groups)):
    D, L = [], []
    r = np.random.RandomState(SEED + fi)
    for u in tr:
        ii, jj = np.triu_indices(J, 1)
        gap = np.abs(Y[u, ii] - Y[u, jj])
        elig = np.where(gap > 0.05)[0]
        for e in r.choice(elig, size=min(12, len(elig)), replace=False):
            i, j = int(ii[e]), int(jj[e])
            D.append(EI[i] - EI[j])
            L.append(1.0 if Y[u, i] > Y[u, j] else 0.0)
    D, L = np.array(D), np.array(L)
    assert D.var() > 0, "zero-diff bug recurring"
    clf = LogisticRegression(max_iter=500).fit(D, L)
    s = np.zeros(J)
    for i in range(J):
        s[i] = np.mean([clf.predict_proba((EI[i] - EI[j]).reshape(1, -1))[0, 1] for j in range(J) if j != i])
    oof_bt[va] = np.tile(s, (len(va), 1))
    fids[va] = fi + 1
    print(f"fold {fi + 1}: pairs={len(L)} train-acc={clf.score(D, L):.4f}", flush=True)

pop = np.tile(Y.mean(0), (n_train, 1))
d = ndcg5(Y, oof_bt) - ndcg5(Y, pop)
lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(2000)], [0.025, 0.975])
wf = min(ndcg5(Y[fids == k], oof_bt[fids == k]).mean() - ndcg5(Y, pop)[fids == k].mean() for k in range(1, N_SPLITS + 1))
print(f"BT OOF={ndcg5(Y, oof_bt).mean():.5f} popularity={ndcg5(Y, pop).mean():.5f} delta={d.mean():+.5f} CI95=[{lo:+.5f},{hi:+.5f}] worst-fold={wf:+.5f}")
print("ADOPT BT strengths" if (lo > GATE_FLOOR and wf >= 0) else "CLOSE pairwise family: module strengths add nothing over popularity")

fold 1: pairs=33672 train-acc=0.6133


fold 2: pairs=40068 train-acc=0.6171


fold 3: pairs=40116 train-acc=0.6185


fold 4: pairs=42144 train-acc=0.6192


fold 5: pairs=36000 train-acc=0.6114


BT OOF=0.38332 popularity=0.39713 delta=-0.01382 CI95=[-0.01667,-0.01071] worst-fold=-0.02088
CLOSE pairwise family: module strengths add nothing over popularity
